# 20 define malignant states

**Purpose:** Define malignant states and calculate transcriptional program scores.

**Original analysis notebook:** `19_malignant_state_composition_program_scoring_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 38 — Malignant-like state composition and program scoring

**Goal:** after final annotation/CNV consensus and differential abundance, focus on the malignant-like compartment and characterize its internal states.

This notebook:
1. maps final clusters to malignant-like states,
2. summarizes malignant-state composition at cell/sample/condition level,
3. runs sample-level Kruskal-Wallis and Mann-Whitney tests,
4. scores GBM-like programs using marker gene modules,
5. creates UMAP/bar/dotplot figures,
6. optionally saves an updated h5ad with malignant-state labels and scores.


In [ ]:
# =========================
# 0. Imports and paths
# =========================

from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import matplotlib.pyplot as plt

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.settings.verbosity = 2

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures" / "final_malignant_state_analysis"

for d in [PROCESSED_DIR, METADATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation.h5ad"

# If disk space is tight, set this to False.
WRITE_UPDATED_H5AD = True
OUTPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"

CLUSTER_KEY = "leiden_scvi_main"
FINAL_COMPARTMENT_COL = "final_compartment_major"
FINAL_ANNOTATION_COL = "final_annotation_refined"
COPYKAT_COL = "copykat_pred_clean"

CONDITION_COL = "condition"
SAMPLE_COL_CANDIDATES = ["sample_id", "sample", "orig.ident", "sample_name"]
PATIENT_COL_CANDIDATES = ["patient_id", "patient", "patient_name"]
DATASET_COL_CANDIDATES = ["core_dataset", "dataset_id", "dataset"]
COUNTS_LAYER_CANDIDATES = ["counts", "raw_counts"]

RANDOM_STATE = 42

print("INPUT_H5AD:", INPUT_H5AD)
print("OUTPUT_H5AD:", OUTPUT_H5AD)


In [ ]:
# =========================
# 1. Helpers
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def savefig_replace(path, dpi=220):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[SAVED FIG] {path}")

def choose_first_existing_column(obs_or_var, candidates, required=False, label="column"):
    for c in candidates:
        if c in obs_or_var.columns:
            return c
    if required:
        raise KeyError(f"Could not find required {label}. Tried: {candidates}")
    return None

def normalize_label(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("/", " ").replace("-", " ")
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    q = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() > 0:
        q[ok] = multipletests(pvals[ok], method="fdr_bh")[1]
    return q

def safe_mannwhitney(x, y):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    y = np.asarray(pd.Series(y).dropna(), dtype=float)
    if len(x) < 1 or len(y) < 1:
        return np.nan
    try:
        return mannwhitneyu(x, y, alternative="two-sided").pvalue
    except Exception:
        return np.nan

def safe_kruskal(groups):
    groups = [np.asarray(pd.Series(g).dropna(), dtype=float) for g in groups if len(pd.Series(g).dropna()) > 0]
    if len(groups) < 2:
        return np.nan
    allv = np.concatenate(groups)
    if len(np.unique(allv)) <= 1:
        return np.nan
    try:
        return kruskal(*groups).pvalue
    except Exception:
        return np.nan

def layer_or_X(adata_obj, candidates):
    for layer in candidates:
        if layer in adata_obj.layers:
            return layer
    return None

def matrix_values_sample(X, n_rows=1000, n_cols=1000):
    Xs = X[:min(n_rows, X.shape[0]), :min(n_cols, X.shape[1])]
    vals = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
    return vals[np.isfinite(vals)]

figure_records = []
manifest_records = []


In [ ]:
# =========================
# 2. Load final annotated object
# =========================

if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Input h5ad not found: {INPUT_H5AD}")

adata = sc.read_h5ad(INPUT_H5AD)
print(adata)

sample_col = choose_first_existing_column(adata.obs, SAMPLE_COL_CANDIDATES, required=True, label="sample column")
patient_col = choose_first_existing_column(adata.obs, PATIENT_COL_CANDIDATES, required=False, label="patient column")
dataset_col = choose_first_existing_column(adata.obs, DATASET_COL_CANDIDATES, required=False, label="dataset column")
counts_layer = layer_or_X(adata, COUNTS_LAYER_CANDIDATES)

required_cols = [CLUSTER_KEY, FINAL_COMPARTMENT_COL, FINAL_ANNOTATION_COL, CONDITION_COL]
missing_cols = [c for c in required_cols if c not in adata.obs.columns]
if missing_cols:
    raise KeyError(f"Missing required obs columns: {missing_cols}")

print("sample_col:", sample_col)
print("patient_col:", patient_col)
print("dataset_col:", dataset_col)
print("counts_layer:", counts_layer)
print("layers:", list(adata.layers.keys())[:20])

vals = matrix_values_sample(adata.layers[counts_layer] if counts_layer else adata.X)
readiness = pd.DataFrame([
    {"item": "n_cells", "value": adata.n_obs},
    {"item": "n_genes", "value": adata.n_vars},
    {"item": "input_h5ad_exists", "value": INPUT_H5AD.exists()},
    {"item": "cluster_key", "value": CLUSTER_KEY},
    {"item": "final_compartment_col", "value": FINAL_COMPARTMENT_COL},
    {"item": "final_annotation_col", "value": FINAL_ANNOTATION_COL},
    {"item": "condition_col", "value": CONDITION_COL},
    {"item": "sample_col", "value": sample_col},
    {"item": "patient_col", "value": patient_col if patient_col else "NA"},
    {"item": "dataset_col", "value": dataset_col if dataset_col else "NA"},
    {"item": "counts_source_for_scoring", "value": counts_layer if counts_layer else "X"},
    {"item": "counts_min_sample", "value": float(vals.min()) if len(vals) else np.nan},
    {"item": "counts_max_sample", "value": float(vals.max()) if len(vals) else np.nan},
    {"item": "counts_integer_like_sample", "value": bool(np.allclose(vals, np.round(vals))) if len(vals) else np.nan},
    {"item": "has_X_umap", "value": "X_umap" in adata.obsm},
])

READINESS_TSV = METADATA_DIR / "malignant_state_readiness_HVG8000_r0_6.tsv"
write_tsv_replace(readiness, READINESS_TSV)
manifest_records.append({"file": str(READINESS_TSV), "description": "Input readiness report"})
display(readiness)


In [ ]:
# =========================
# 3. Define malignant-like states from final clusters
# =========================

# Consensus definitions from marker annotation + CopyKAT support.
CLUSTER_TO_MALIGNANT_STATE = {
    "0": "OPC_NPC_like_malignant",
    "4": "hypoxia_stress_malignant",
    "7": "AC_like_reactive_malignant",
    "11": "NPC_neuronal_like_malignant",
    "2": "cycling_malignant_candidate",
}

adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str)
adata.obs["malignant_state"] = adata.obs[CLUSTER_KEY].map(CLUSTER_TO_MALIGNANT_STATE).fillna("not_malignant_state")
adata.obs["is_malignant_state_analysis_cell"] = adata.obs["malignant_state"].ne("not_malignant_state")
adata.obs["_final_compartment_norm"] = adata.obs[FINAL_COMPARTMENT_COL].map(normalize_label)

mal_state_counts = (
    adata.obs["malignant_state"]
    .value_counts(dropna=False)
    .rename_axis("malignant_state")
    .reset_index(name="n_cells")
)
mal_state_counts["fraction_all_cells"] = mal_state_counts["n_cells"] / adata.n_obs

STATE_COUNTS_TSV = METADATA_DIR / "malignant_state_cell_counts_HVG8000_r0_6.tsv"
write_tsv_replace(mal_state_counts, STATE_COUNTS_TSV)
manifest_records.append({"file": str(STATE_COUNTS_TSV), "description": "Cell counts for malignant-state labels"})
display(mal_state_counts)


In [ ]:
# =========================
# 4. Cluster-to-state QC summary
# =========================

qc_cols = [CLUSTER_KEY, "malignant_state", FINAL_COMPARTMENT_COL, FINAL_ANNOTATION_COL]
cluster_state_summary = (
    adata.obs[qc_cols]
    .groupby([CLUSTER_KEY, "malignant_state", FINAL_COMPARTMENT_COL, FINAL_ANNOTATION_COL], observed=True)
    .size()
    .reset_index(name="n_cells")
)

if COPYKAT_COL in adata.obs.columns:
    ck = (
        adata.obs.groupby([CLUSTER_KEY, COPYKAT_COL], observed=True)
        .size()
        .reset_index(name="n")
    )
    ck_tot = ck.groupby(CLUSTER_KEY, observed=True)["n"].sum().reset_index(name="copykat_total")
    ck = ck.merge(ck_tot, on=CLUSTER_KEY, how="left")
    ck["fraction"] = ck["n"] / ck["copykat_total"]
    ck_piv = ck.pivot_table(index=CLUSTER_KEY, columns=COPYKAT_COL, values="fraction", fill_value=0).reset_index()
    ck_piv.columns = [str(c) if c == CLUSTER_KEY else f"copykat_fraction_{c}" for c in ck_piv.columns]
    cluster_state_summary = cluster_state_summary.merge(ck_piv, on=CLUSTER_KEY, how="left")

CLUSTER_STATE_TSV = METADATA_DIR / "malignant_state_cluster_summary_HVG8000_r0_6.tsv"
write_tsv_replace(cluster_state_summary, CLUSTER_STATE_TSV)
manifest_records.append({"file": str(CLUSTER_STATE_TSV), "description": "Cluster-to-malignant-state QC summary"})
display(cluster_state_summary)


In [ ]:
# =========================
# 5. Sample-level malignant-state composition
# =========================

mal_cells = adata.obs[adata.obs["is_malignant_state_analysis_cell"]].copy()

id_cols = [sample_col, CONDITION_COL]
if patient_col:
    id_cols.append(patient_col)
if dataset_col:
    id_cols.append(dataset_col)

state_counts = (
    mal_cells.groupby(id_cols + ["malignant_state"], observed=True)
    .size()
    .reset_index(name="n_state_cells")
)
sample_totals = (
    mal_cells.groupby(id_cols, observed=True)
    .size()
    .reset_index(name="n_malignant_state_analysis_cells")
)
total_sample_cells = (
    adata.obs.groupby(id_cols, observed=True)
    .size()
    .reset_index(name="n_total_cells_in_sample")
)

all_samples = sample_totals[id_cols].drop_duplicates()
all_states = pd.DataFrame({"malignant_state": sorted(CLUSTER_TO_MALIGNANT_STATE.values())})
all_grid = all_samples.merge(all_states, how="cross")

sample_state = all_grid.merge(state_counts, on=id_cols + ["malignant_state"], how="left")
sample_state = sample_state.merge(sample_totals, on=id_cols, how="left")
sample_state = sample_state.merge(total_sample_cells, on=id_cols, how="left")

sample_state["n_state_cells"] = sample_state["n_state_cells"].fillna(0).astype(int)
sample_state["fraction_within_malignant_state_pool"] = (
    sample_state["n_state_cells"] / sample_state["n_malignant_state_analysis_cells"]
)
sample_state["fraction_of_all_sample_cells"] = (
    sample_state["n_state_cells"] / sample_state["n_total_cells_in_sample"]
)

SAMPLE_STATE_TSV = METADATA_DIR / "malignant_state_sample_level_proportions_HVG8000_r0_6.tsv"
write_tsv_replace(sample_state, SAMPLE_STATE_TSV)
manifest_records.append({"file": str(SAMPLE_STATE_TSV), "description": "Sample-level malignant state proportions"})
display(sample_state.head())
print("n samples:", sample_state[sample_col].nunique())


In [ ]:
# =========================
# 6. Condition summary for malignant states
# =========================

summary_rows = []
for state, sub in sample_state.groupby("malignant_state", observed=True):
    for condition, g in sub.groupby(CONDITION_COL, observed=True):
        vals = g["fraction_within_malignant_state_pool"].dropna().astype(float)
        summary_rows.append({
            "malignant_state": state,
            "condition": condition,
            "n_samples": int(vals.shape[0]),
            "mean_fraction": float(vals.mean()) if len(vals) else np.nan,
            "median_fraction": float(vals.median()) if len(vals) else np.nan,
            "q25_fraction": float(vals.quantile(0.25)) if len(vals) else np.nan,
            "q75_fraction": float(vals.quantile(0.75)) if len(vals) else np.nan,
            "min_fraction": float(vals.min()) if len(vals) else np.nan,
            "max_fraction": float(vals.max()) if len(vals) else np.nan,
        })

condition_summary = pd.DataFrame(summary_rows)

CONDITION_SUMMARY_TSV = METADATA_DIR / "malignant_state_condition_summary_sample_level_HVG8000_r0_6.tsv"
write_tsv_replace(condition_summary, CONDITION_SUMMARY_TSV)
manifest_records.append({"file": str(CONDITION_SUMMARY_TSV), "description": "Condition-level summary of malignant-state sample fractions"})
display(condition_summary)


In [ ]:
# =========================
# 7. Kruskal-Wallis and pairwise Mann-Whitney tests
# =========================

test_col = "fraction_within_malignant_state_pool"

kw_rows = []
for state, sub in sample_state.groupby("malignant_state", observed=True):
    groups = []
    group_names = []
    for condition, g in sub.groupby(CONDITION_COL, observed=True):
        vals = g[test_col].dropna().astype(float)
        if len(vals) > 0:
            groups.append(vals)
            group_names.append(condition)
    p = safe_kruskal(groups)
    kw_rows.append({
        "malignant_state": state,
        "test": "Kruskal-Wallis",
        "value_tested": test_col,
        "conditions_tested": ",".join(map(str, group_names)),
        "n_conditions": len(group_names),
        "p_value": p,
    })

kw_df = pd.DataFrame(kw_rows)
kw_df["q_value_bh"] = bh_fdr(kw_df["p_value"])

KW_TSV = METADATA_DIR / "malignant_state_global_kruskal_sample_level_HVG8000_r0_6.tsv"
write_tsv_replace(kw_df, KW_TSV)
manifest_records.append({"file": str(KW_TSV), "description": "Kruskal-Wallis tests for malignant-state proportions by condition"})
display(kw_df.sort_values("q_value_bh"))


In [ ]:
# Pairwise Mann-Whitney U tests
pair_rows = []
conditions = sorted(sample_state[CONDITION_COL].dropna().astype(str).unique())

for state, sub in sample_state.groupby("malignant_state", observed=True):
    sub = sub.copy()
    sub[CONDITION_COL] = sub[CONDITION_COL].astype(str)
    for i in range(len(conditions)):
        for j in range(i + 1, len(conditions)):
            c1, c2 = conditions[i], conditions[j]
            x = sub.loc[sub[CONDITION_COL] == c1, test_col].astype(float)
            y = sub.loc[sub[CONDITION_COL] == c2, test_col].astype(float)
            p = safe_mannwhitney(x, y)
            pair_rows.append({
                "malignant_state": state,
                "test": "Mann-Whitney U",
                "value_tested": test_col,
                "condition_1": c1,
                "condition_2": c2,
                "n_samples_1": int(x.dropna().shape[0]),
                "n_samples_2": int(y.dropna().shape[0]),
                "median_1": float(x.median()) if len(x.dropna()) else np.nan,
                "median_2": float(y.median()) if len(y.dropna()) else np.nan,
                "delta_median_1_minus_2": float(x.median() - y.median()) if len(x.dropna()) and len(y.dropna()) else np.nan,
                "p_value": p,
            })

pair_df = pd.DataFrame(pair_rows)
pair_df["q_value_bh"] = bh_fdr(pair_df["p_value"])

PAIR_TSV = METADATA_DIR / "malignant_state_pairwise_mannwhitney_sample_level_HVG8000_r0_6.tsv"
write_tsv_replace(pair_df, PAIR_TSV)
manifest_records.append({"file": str(PAIR_TSV), "description": "Pairwise Mann-Whitney tests for malignant-state proportions by condition"})
display(pair_df.sort_values("q_value_bh").head(30))


In [ ]:
# =========================
# 8. Program marker panels
# =========================

PROGRAM_MARKERS = {
    "OPC_proneural_program": ["PDGFRA", "OLIG1", "OLIG2", "PTPRZ1", "BCAN", "SOX6", "SOX10", "CSPG4", "NKX2-2"],
    "NPC_neural_program": ["SOX4", "SOX11", "DCX", "STMN2", "TUBB3", "DLX5", "MYT1L", "ASCL1", "NEUROD1"],
    "AC_like_reactive_program": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "CLU", "SPARCL1", "APOE", "VIM"],
    "MES_like_program": ["CHI3L1", "CD44", "VIM", "FN1", "SERPINE1", "TGFBI", "LGALS3", "CCL2", "ITGA5"],
    "hypoxia_stress_program": ["CA9", "VEGFA", "HILPDA", "LOX", "BNIP3", "PDK1", "SLC2A1", "DDIT4", "NDRG1"],
    "cycling_program": ["MKI67", "TOP2A", "PBK", "BIRC5", "UBE2C", "CENPF", "NUSAP1", "CENPA", "HMGB2"],
    "myeloid_contamination_review_program": ["C1QA", "C1QB", "C1QC", "TYROBP", "LYZ", "FCN1", "CD74", "HLA-DRA", "CSF1R"],
}

marker_rows = []
for program, genes in PROGRAM_MARKERS.items():
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    marker_rows.append({
        "program": program,
        "n_genes_requested": len(genes),
        "n_genes_present": len(present),
        "n_genes_missing": len(missing),
        "present_genes": ",".join(present),
        "missing_genes": ",".join(missing),
    })

marker_presence = pd.DataFrame(marker_rows)
MARKER_PRESENCE_TSV = METADATA_DIR / "malignant_state_program_marker_presence_HVG8000_r0_6.tsv"
write_tsv_replace(marker_presence, MARKER_PRESENCE_TSV)
manifest_records.append({"file": str(MARKER_PRESENCE_TSV), "description": "Presence/absence of malignant program marker genes"})
display(marker_presence)


In [ ]:
# =========================
# 9. Score malignant-state programs
# =========================

mal_mask = adata.obs["is_malignant_state_analysis_cell"].values
adata_score = adata[mal_mask].copy()
print(adata_score)

score_counts_layer = layer_or_X(adata_score, COUNTS_LAYER_CANDIDATES)
if score_counts_layer is not None:
    adata_score.X = adata_score.layers[score_counts_layer].copy()
    print("Using counts layer for scoring:", score_counts_layer)
else:
    print("No counts layer found; using X for scoring.")

sc.pp.normalize_total(adata_score, target_sum=1e4)
sc.pp.log1p(adata_score)

score_cols = []
for program, genes in PROGRAM_MARKERS.items():
    present = [g for g in genes if g in adata_score.var_names]
    score_name = f"score_{program}"
    if len(present) >= 2:
        sc.tl.score_genes(
            adata_score,
            gene_list=present,
            score_name=score_name,
            random_state=RANDOM_STATE,
            use_raw=False,
        )
        print("[SCORED]", score_name, "n_genes=", len(present))
    else:
        adata_score.obs[score_name] = np.nan
        print("[SKIPPED]", score_name, "not enough genes present")
    score_cols.append(score_name)

for col in score_cols:
    adata.obs[col] = np.nan
    adata.obs.loc[adata_score.obs_names, col] = adata_score.obs[col].astype(float)

print("score_cols:", score_cols)


In [ ]:
# =========================
# 10. Score summaries
# =========================

score_df = adata.obs.loc[adata.obs["is_malignant_state_analysis_cell"], [
    "malignant_state", CONDITION_COL, CLUSTER_KEY, sample_col
] + score_cols].copy()

score_summary_rows = []
for state, sub_state in score_df.groupby("malignant_state", observed=True):
    for score_col in score_cols:
        vals = sub_state[score_col].dropna().astype(float)
        score_summary_rows.append({
            "level": "malignant_state",
            "malignant_state": state,
            "condition": "all",
            "score": score_col,
            "n_cells": int(vals.shape[0]),
            "mean_score": float(vals.mean()) if len(vals) else np.nan,
            "median_score": float(vals.median()) if len(vals) else np.nan,
            "q25_score": float(vals.quantile(0.25)) if len(vals) else np.nan,
            "q75_score": float(vals.quantile(0.75)) if len(vals) else np.nan,
        })
    for condition, sub_cond in sub_state.groupby(CONDITION_COL, observed=True):
        for score_col in score_cols:
            vals = sub_cond[score_col].dropna().astype(float)
            score_summary_rows.append({
                "level": "malignant_state_by_condition",
                "malignant_state": state,
                "condition": condition,
                "score": score_col,
                "n_cells": int(vals.shape[0]),
                "mean_score": float(vals.mean()) if len(vals) else np.nan,
                "median_score": float(vals.median()) if len(vals) else np.nan,
                "q25_score": float(vals.quantile(0.25)) if len(vals) else np.nan,
                "q75_score": float(vals.quantile(0.75)) if len(vals) else np.nan,
            })

score_summary = pd.DataFrame(score_summary_rows)
SCORE_SUMMARY_TSV = METADATA_DIR / "malignant_state_program_score_summary_HVG8000_r0_6.tsv"
write_tsv_replace(score_summary, SCORE_SUMMARY_TSV)
manifest_records.append({"file": str(SCORE_SUMMARY_TSV), "description": "Program score summaries by malignant state and condition"})
display(score_summary.head(30))


In [ ]:
# =========================
# 11. Sample-level program scores
# =========================

sample_score_rows = []
for (sample, state), sub in score_df.groupby([sample_col, "malignant_state"], observed=True):
    row = {
        sample_col: sample,
        "malignant_state": state,
        "n_cells": int(sub.shape[0]),
        CONDITION_COL: sub[CONDITION_COL].iloc[0],
    }
    if patient_col and patient_col in adata.obs.columns:
        row[patient_col] = adata.obs.loc[sub.index, patient_col].iloc[0]
    if dataset_col and dataset_col in adata.obs.columns:
        row[dataset_col] = adata.obs.loc[sub.index, dataset_col].iloc[0]
    for score_col in score_cols:
        row[f"mean_{score_col}"] = float(sub[score_col].mean()) if sub[score_col].notna().sum() else np.nan
        row[f"median_{score_col}"] = float(sub[score_col].median()) if sub[score_col].notna().sum() else np.nan
    sample_score_rows.append(row)

sample_score_summary = pd.DataFrame(sample_score_rows)
SAMPLE_SCORE_TSV = METADATA_DIR / "malignant_state_sample_level_program_scores_HVG8000_r0_6.tsv"
write_tsv_replace(sample_score_summary, SAMPLE_SCORE_TSV)
manifest_records.append({"file": str(SAMPLE_SCORE_TSV), "description": "Sample-level program score summaries for malignant states"})
display(sample_score_summary.head())


In [ ]:
# =========================
# 12. Figures
# =========================

if "X_umap" in adata.obsm:
    sc.pl.umap(
        adata,
        color=["malignant_state", FINAL_COMPARTMENT_COL, CONDITION_COL],
        wspace=0.35,
        show=False,
    )
    fig_path = FIGURES_DIR / "UMAP_malignant_state_final_compartment_condition_HVG8000_r0_6.png"
    savefig_replace(fig_path)
    figure_records.append({"figure": str(fig_path), "description": "UMAP colored by malignant state, final compartment and condition"})

    for col in score_cols:
        sc.pl.umap(
            adata,
            color=col,
            show=False,
            cmap="viridis",
            vmin="p1",
            vmax="p99",
        )
        fig_path = FIGURES_DIR / f"UMAP_{col}_HVG8000_r0_6.png"
        savefig_replace(fig_path)
        figure_records.append({"figure": str(fig_path), "description": f"UMAP program score: {col}"})
else:
    print("[WARNING] X_umap not found; skipping UMAP figures.")

# Barplot: sample-level median malignant state fractions by condition
plot_df = condition_summary.copy()
fig, ax = plt.subplots(figsize=(10, 5))
states = sorted(plot_df["malignant_state"].unique())
x = np.arange(len(states))
conds = sorted(plot_df["condition"].astype(str).unique())
width = min(0.8 / max(len(conds), 1), 0.18)

for idx, cond in enumerate(conds):
    vals = []
    for state in states:
        row = plot_df[(plot_df["malignant_state"] == state) & (plot_df["condition"].astype(str) == cond)]
        vals.append(float(row["median_fraction"].iloc[0]) if len(row) else 0)
    ax.bar(x + (idx - (len(conds)-1)/2) * width, vals, width, label=cond)

ax.set_xticks(x)
ax.set_xticklabels(states, rotation=45, ha="right")
ax.set_ylabel("Median fraction within malignant-state pool")
ax.set_title("Malignant-state composition by condition")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig_path = FIGURES_DIR / "bar_malignant_state_condition_median_sample_fraction_HVG8000_r0_6.png"
savefig_replace(fig_path)
figure_records.append({"figure": str(fig_path), "description": "Barplot of sample-level median malignant-state fractions by condition"})


In [ ]:
# =========================
# 13. Dotplot marker panel by malignant state
# =========================

dotplot_markers = []
for program, genes in PROGRAM_MARKERS.items():
    present = [g for g in genes if g in adata_score.var_names]
    dotplot_markers.extend(present[:6])

seen = set()
dotplot_markers = [g for g in dotplot_markers if not (g in seen or seen.add(g))]

if len(dotplot_markers) > 0:
    sc.pl.dotplot(
        adata_score,
        var_names=dotplot_markers,
        groupby="malignant_state",
        standard_scale="var",
        show=False,
    )
    fig_path = FIGURES_DIR / "dotplot_malignant_state_program_markers_HVG8000_r0_6.png"
    savefig_replace(fig_path)
    figure_records.append({"figure": str(fig_path), "description": "Dotplot of malignant-state program markers"})
else:
    print("[WARNING] No dotplot markers found.")


In [ ]:
# =========================
# 14. Optional save updated h5ad
# =========================

if WRITE_UPDATED_H5AD:
    if OUTPUT_H5AD.exists():
        OUTPUT_H5AD.unlink()
    adata.write_h5ad(OUTPUT_H5AD)
    print("[SAVED H5AD]", OUTPUT_H5AD)
    manifest_records.append({"file": str(OUTPUT_H5AD), "description": "Updated h5ad with malignant-state labels and program scores"})
else:
    print("[SKIPPED] WRITE_UPDATED_H5AD is False")


In [ ]:
# =========================
# 15. Manifests
# =========================

fig_manifest = pd.DataFrame(figure_records)
FIG_MANIFEST_TSV = METADATA_DIR / "malignant_state_figure_manifest_HVG8000_r0_6.tsv"
write_tsv_replace(fig_manifest, FIG_MANIFEST_TSV)
manifest_records.append({"file": str(FIG_MANIFEST_TSV), "description": "Figure manifest for malignant-state analysis"})

out_manifest = pd.DataFrame(manifest_records)
OUT_MANIFEST_TSV = METADATA_DIR / "malignant_state_analysis_output_manifest_HVG8000_r0_6.tsv"
write_tsv_replace(out_manifest, OUT_MANIFEST_TSV)

display(out_manifest)
print("[DONE] Malignant-like state composition and program scoring complete.")
